# 🎯 Module 13 · 07 — Text Classification

> **Stage:** Machine Learning → NLP  
> **Level:** Intermediate  
> **Module:** 13 — NLP Basics / 07 — Text Classification  
> **Date:** 31 January 2026  
> **Previous:** 06 — Word Embeddings  
> **Next:** 08 — Sentiment Analysis

---

**Text classification** is the process of assigning text to one or more predefined **categories**.

In this notebook we'll cover:

- What text classification is
- **Supervised learning** framing
- Binary / multiclass / multi-label classification
- Common classifiers: **Naive Bayes**, **Logistic Regression**, **Linear SVM**
- Train/test split & **stratification**
- **Data leakage** & how to avoid it
- Scikit-learn **Pipeline**
- **Evaluation** — accuracy, precision, recall, F1
- **Confusion matrix**
- **Class imbalance**
- **Feature engineering** — word vs character n-grams
- **Baseline first**
- **Hyperparameter tuning**
- **Error analysis**

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain what text classification is
- Distinguish **binary**, **multiclass**, and **multi-label** classification
- Build a text classifier with a **Pipeline**
- Use **Naive Bayes**, **Logistic Regression**, and **Linear SVM**
- Split data with **stratification**
- Avoid **data leakage**
- Evaluate with accuracy, precision, recall, F1, and **confusion matrix**
- Handle **class imbalance**
- Use **word** and **character n-gram** features
- Follow a **baseline-first** workflow
- **Tune** hyperparameters
- Run **error analysis**

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)

np.random.seed(42)
print('✅ Setup ready.')

---
## 🧠 1. What Is Text Classification?

Given:

```text
"This product is amazing"
```

and categories:

```text
Positive
Negative
```

the model predicts:

```text
"This product is amazing"  →  Positive
```

### Labeled training data

```text
Text                            Label
-------------------------------------
"This is excellent"             Positive
"I love this product"           Positive
"This is terrible"              Negative
"I hate this product"           Negative
```

---
## 🎓 2. Supervised Learning

Classical text classification is a **supervised learning** problem.

```text
X = Text
y = Labels
```

Example:

```text
X:  ["I love this",  "This is terrible"]
y:  ["positive",     "negative"]
```

The model learns:

```text
Text features → Label
```

Then it can label **unseen** text.

---
## 🔢 3. Text Must Become Numbers

Models can't process raw strings — we must convert to numbers.

Common representations:

```text
Bag of Words
TF-IDF
Word Embeddings
```

### Classic baseline

```text
Text
 ↓
TF-IDF
 ↓
Logistic Regression
 ↓
Prediction
```

---
## 🔧 4. Common Classification Algorithms

### Naive Bayes

```text
Text
 ↓
BoW / TF-IDF
 ↓
Naive Bayes
 ↓
Class
```

Fast, simple, strong baseline for text.

### Logistic Regression

```text
TF-IDF
 ↓
Logistic Regression
 ↓
Prediction
```

Strong, interpretable baseline.

### Linear SVM

```text
TF-IDF
 ↓
Linear SVM
 ↓
Prediction
```

Often performs very well on **sparse high-dimensional** text features.

> Pick the winner on **validation**, not by guessing.

---
## 2️⃣ 5. Binary Classification

Two classes:

```text
Spam / Not Spam
Positive / Negative
```

Example:

```text
"Congratulations! You won a prize."
          ↓
         Spam
```

The model separates the two classes based on features.

---
## 3️⃣ 6. Multiclass Classification

More than two classes — one document → **one** class.

```text
News
 ├── Sports
 ├── Technology
 ├── Business
 └── Entertainment
```

Example:

```text
"Apple released a new processor"
              ↓
         Technology
```

---
## 🏷️ 7. Multi-Label Classification

A document can receive **multiple labels**.

Example:

```text
"AI is being used in healthcare and robotics."
```

Possible labels:

```text
AI
Healthcare
Robotics
```

### The distinction

```text
Multiclass   -> one class per document
Multi-label  -> multiple classes per document
```

---
## ✂️ 8. Train / Test Split

```text
Dataset
   ↓
Train / Test Split
   ↓
Training Data  → Fit Vectorizer + Model
   ↓
Testing Data   → Transform + Predict
   ↓
Evaluation
```

### Stratification

For classification, use **stratified** splits to preserve class proportions:

```python
train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
```

---
## 🚫 9. Data Leakage

> **Never allow information from the test set to influence training.**

### ❌ Incorrect

```text
Entire Dataset
      ↓
Fit TF-IDF
      ↓
Train/Test Split
```

### ✅ Correct

```text
Dataset
   ↓
Train/Test Split
   ↓
Training Text → Fit TF-IDF
   ↓
Testing Text  → Transform Only
```

A scikit-learn **Pipeline** enforces this automatically.

---
## 🏗️ 10. Scikit-Learn Pipeline

Combine vectorization + model into one object:

```python
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', LogisticRegression()),
])
```

Then:

```python
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)
```

The pipeline ensures the same transformation is applied **consistently** and prevents leakage.

---
## 🧪 11. Complete Example

```python
texts = [
    "I love this product", "This is amazing", "This product is excellent",
    "I hate this product", "This is terrible", "Very bad experience",
]
labels = ["positive", "positive", "positive", "negative", "negative", "negative"]

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', LogisticRegression()),
])

pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)
```

### Architecture

```text
Text → TF-IDF → Logistic Regression → Prediction
```

In [ ]:
# Build a synthetic sentiment dataset for full demo
positive = [
    "I love this product", "This is amazing", "This product is excellent",
    "Absolutely wonderful", "Best purchase ever", "Highly recommend",
    "Fantastic quality", "Great service", "Very happy with this",
    "Superb experience", "Exceeded expectations", "Really enjoyed it",
    "Amazing quality", "I'm impressed", "Five stars",
]
negative = [
    "I hate this product", "This is terrible", "Very bad experience",
    "Absolutely awful", "Worst purchase ever", "Do not recommend",
    "Horrible quality", "Bad service", "Very disappointed with this",
    "Waste of money", "Below expectations", "Really disliked it",
    "Poor quality", "I'm frustrated", "One star",
]

texts  = positive + negative
labels = ['positive'] * len(positive) + ['negative'] * len(negative)

print(f'Dataset size: {len(texts)}')
print(f'Positive: {len(positive)}  Negative: {len(negative)}')

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels
)

print(f'\nTrain: {len(X_train)}  Test: {len(X_test)}')

In [ ]:
# Baseline pipeline: TF-IDF + Logistic Regression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])

pipeline.fit(X_train, y_train)
preds = pipeline.predict(X_test)

print('Baseline results:')
print(f'  Accuracy : {accuracy_score(y_test, preds):.3f}')
print(f'  Precision: {precision_score(y_test, preds, average="macro"):.3f}')
print(f'  Recall   : {recall_score(y_test, preds, average="macro"):.3f}')
print(f'  F1       : {f1_score(y_test, preds, average="macro"):.3f}')

In [ ]:
# Predict a new sentence
new_texts = [
    "This product is absolutely incredible",
    "I really regret buying this",
]
print('New predictions:')
for text, pred in zip(new_texts, pipeline.predict(new_texts)):
    print(f'  {text!r:<45} -> {pred}')

---
## 📊 12. Evaluation

Don't judge only by accuracy.

Important metrics:

```text
Accuracy
Precision
Recall
F1 Score
ROC-AUC
```

### Precision

$$\text{Precision} = \frac{TP}{TP + FP}$$

Of everything predicted positive, how many were actually positive?

### Recall

$$\text{Recall} = \frac{TP}{TP + FN}$$

Of all actual positives, how many did we find?

### F1 Score

$$F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

Balances precision and recall.

In [ ]:
# Full classification report
print(classification_report(y_test, preds, digits=3))

---
## 🔢 13. Confusion Matrix

```text
                  Predicted
                Positive  Negative

Actual Positive    TP        FN
Actual Negative    FP        TN
```

- **TP** → True Positive
- **TN** → True Negative
- **FP** → False Positive
- **FN** → False Negative

This shows **what kinds of mistakes** the model makes.

In [ ]:
# Confusion matrix heatmap
labels_order = pipeline.classes_
cm = confusion_matrix(y_test, preds, labels=labels_order)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels_order)))
ax.set_yticks(range(len(labels_order)))
ax.set_xticklabels(labels_order)
ax.set_yticklabels(labels_order)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
for i in range(len(labels_order)):
    for j in range(len(labels_order)):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=13)
plt.colorbar(im)
plt.title('Confusion matrix')
plt.tight_layout(); plt.show()

---
## ⚖️ 14. Class Imbalance

If classes are very different in size:

```text
Normal messages -> 9,500
Spam messages   ->   500
```

The model can hit high accuracy by predicting **Normal** almost always.

### Solutions

- **Class weights**
- Better metrics (F1, PR-AUC)
- Resampling
- More representative data

### Example — balanced class weights

```python
LogisticRegression(class_weight='balanced')
```

In [ ]:
# Simulate class imbalance
from sklearn.datasets import make_classification
import random

random.seed(42)

# Build an imbalanced synthetic corpus: many "normal", few "spam"
normal_msgs = [
    "Hi, how are you?", "Let's meet tomorrow", "The report is ready",
    "Please review this document", "Meeting at 3 PM",
    "Thanks for your help", "Let me know when you're free",
    "Check your inbox", "I'll call you later",
]
spam_msgs = [
    "Win a free prize now!!!", "Congratulations! You won $1000",
    "Claim your reward today",
]

# Duplicate to create imbalance
texts_imb = normal_msgs * 20 + spam_msgs * 5
labels_imb = ['ham'] * (len(normal_msgs) * 20) + ['spam'] * (len(spam_msgs) * 5)

print('Class distribution:', pd.Series(labels_imb).value_counts().to_dict())

X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(
    texts_imb, labels_imb, test_size=0.2, random_state=42, stratify=labels_imb
)

# Compare: default vs balanced
for cw in [None, 'balanced']:
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('clf', LogisticRegression(max_iter=1000, class_weight=cw, random_state=42)),
    ])
    pipe.fit(X_tr_i, y_tr_i)
    p = pipe.predict(X_te_i)
    tag = 'default  ' if cw is None else 'balanced '
    print(f'{tag} accuracy={accuracy_score(y_te_i, p):.3f}  f1(macro)={f1_score(y_te_i, p, average="macro"):.3f}')

---
## 🛠️ 15. Feature Engineering

Classical NLP features:

```text
Word features
Character features
N-grams
TF-IDF
Binary features
```

### Word features
Best when word **content** carries most meaning.

### Character features
Useful for:

- Spelling variations
- URLs / usernames
- Product codes
- Short text
- Noisy text

```python
TfidfVectorizer(analyzer='char', ngram_range=(3, 5))
```

In [ ]:
# Compare word vs character n-grams on the same task
configs = {
    'word (1,1)':       TfidfVectorizer(analyzer='word', ngram_range=(1, 1)),
    'word (1,2)':       TfidfVectorizer(analyzer='word', ngram_range=(1, 2)),
    'char (2,4)':       TfidfVectorizer(analyzer='char', ngram_range=(2, 4)),
    'char_wb (3,5)':    TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5)),
}

print(f'{"Config":<16} {"Vocab":>8} {"Acc":>8} {"F1":>8}')
print('-' * 44)
for name, vec in configs.items():
    pipe = Pipeline([
        ('vec', vec),
        ('clf', LogisticRegression(max_iter=1000, random_state=42)),
    ])
    pipe.fit(X_train, y_train)
    p = pipe.predict(X_test)
    v = len(pipe.named_steps['vec'].get_feature_names_out())
    print(f'{name:<16} {v:>8} {accuracy_score(y_test, p):>8.3f} {f1_score(y_test, p, average="macro"):>8.3f}')

---
## 📏 16. Baseline First

```text
Dataset
 ↓
Simple preprocessing
 ↓
TF-IDF
 ↓
Simple model
 ↓
Evaluation
```

A simple baseline like:

```text
TF-IDF + Logistic Regression
```

Then compare:

```text
TF-IDF + Naive Bayes
TF-IDF + Logistic Regression
TF-IDF + Linear SVM
```

This makes model selection **evidence-based** instead of guessing.

In [ ]:
# Compare NB, LR, and LinearSVC on the same data
classifiers = {
    'Naive Bayes': MultinomialNB(),
    'Logistic':    LogisticRegression(max_iter=1000, random_state=42),
    'LinearSVC':   LinearSVC(random_state=42),
}

print(f'{"Model":<14} {"Acc":>8} {"Prec":>8} {"Rec":>8} {"F1":>8}')
print('-' * 48)
for name, clf in classifiers.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('clf', clf),
    ])
    pipe.fit(X_train, y_train)
    p = pipe.predict(X_test)
    print(f'{name:<14} '
          f'{accuracy_score(y_test, p):>8.3f} '
          f'{precision_score(y_test, p, average="macro"):>8.3f} '
          f'{recall_score(y_test, p, average="macro"):>8.3f} '
          f'{f1_score(y_test, p, average="macro"):>8.3f}')

---
## 🎛️ 17. Hyperparameter Tuning

Tune the pipeline with **GridSearchCV**:

```python
parameters = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df':      [1, 2],
    'classifier__C':      [0.1, 1, 10],
}

grid = GridSearchCV(pipeline, parameters, cv=5, scoring='f1')
grid.fit(X_train, y_train)
```

Uses **cross-validation** on training data — the test set is never touched.

In [ ]:
# Small grid search on the sentiment dataset
pipe = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])

params = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df':      [1, 2],
    'classifier__C':      [0.1, 1, 10],
}

grid = GridSearchCV(pipe, params, cv=3, scoring='f1_macro', n_jobs=-1)
grid.fit(X_train, y_train)

print('Best params:', grid.best_params_)
print(f'Best CV F1 : {grid.best_score_:.4f}')

best_preds = grid.predict(X_test)
print(f'Test F1    : {f1_score(y_test, best_preds, average="macro"):.4f}')

---
## 🔍 18. Error Analysis

A single number doesn't tell the whole story — inspect **wrong predictions**.

### Examples

```text
"This phone is sick 🔥"
Actual: Positive  |  Predicted: Negative
```
Reason: **slang / emoji**.

```text
"I expected better, but the battery is excellent."
Predicted: Positive
```
Reason: **mixed sentiment**.

Error analysis reveals **what to fix next**.

In [ ]:
# Show wrong predictions on the test set
preds = grid.predict(X_test)
wrong = [(t, a, p) for t, a, p in zip(X_test, y_test, preds) if a != p]

print(f'Total test samples: {len(X_test)}')
print(f'Misclassified     : {len(wrong)}')
print()
if wrong:
    print('Examples of wrong predictions:')
    for text, actual, pred in wrong[:5]:
        print(f'  {text!r}')
        print(f'    actual={actual}  predicted={pred}')
else:
    print('No misclassifications on this small test set (uncommon but possible).')

---
## 🌍 19. Real-World Applications

Text classification is used for:

```text
Spam Detection
Sentiment Analysis
News Classification
Customer Support Routing
Intent Classification
Toxicity Detection
Resume Classification
Email Categorization
Topic Classification
```

### Customer support example

```text
Customer Message
       ↓
TF-IDF
       ↓
Classifier
       ↓
┌─────────────────┐
│ Refund          │
│ Technical Issue │
│ Order Status    │
│ Complaint       │
└─────────────────┘
```

---
## 🧠 20. Mental Model

```text
                  TEXT
                   ↓
             PREPROCESSING
                   ↓
          TEXT REPRESENTATION
          ┌────────┴────────┐
        BoW             TF-IDF
          └────────┬────────┘
                   ↓
             ML CLASSIFIER
          ┌──────┬──────┬─────┐
          NB    LR     SVM
          └──────┴──────┴─────┘
                   ↓
              PREDICTION
                   ↓
               EVALUATION
```

### 🧭 Golden Rule

> **Text classification combines text preprocessing, numerical representation, and supervised machine learning to assign text to meaningful categories.**

---
## 🏋️ 21. Hands-On Exercises

### Exercise 1 — Binary classifier
Build a spam / ham classifier on this small dataset:
```python
texts = [
    "Win a free prize now", "Claim your reward",
    "Let's meet tomorrow", "Report is ready",
]
labels = ['spam', 'spam', 'ham', 'ham']
```
Report accuracy and F1.

### Exercise 2 — Multiclass classifier
Build a classifier for these 3 categories:
- `sports`
- `tech`
- `politics`

Use at least 5 examples per class.

### Exercise 3 — Data leakage demo
Write two pieces of code:
1. Fit TF-IDF **before** the split (wrong)
2. Fit TF-IDF only on training data (right)

Compare the reported accuracy.

### Exercise 4 — Class imbalance
Create an imbalanced dataset (1:20 ratio).  
Compare plain LogisticRegression vs `class_weight='balanced'`.

### Exercise 5 — Word vs char features
Compare word TF-IDF and char TF-IDF on:
```text
"I luv this product!!!!",
"This is gr8 quality",
"Absolutely amazing!!",
```
Which handles noisy text better?

### Exercise 6 — Error analysis
Train a classifier and list all misclassified examples.  
Group them into categories (slang, negation, mixed sentiment, etc.).

---
## 🎤 22. Interview Questions

1. **What is text classification?**  
   Assigning text to predefined categories using features learned from labeled data.

2. **Is text classification supervised or unsupervised?**  
   Typically supervised — the model learns from labeled examples.

3. **Binary vs multiclass vs multi-label?**  
   Two classes; many classes (one per doc); many classes (multiple per doc).

4. **Why TF-IDF + Logistic Regression?**  
   It's a strong, fast, interpretable baseline for sparse text features.

5. **When does Naive Bayes work well?**  
   Small text datasets, spam detection, and when speed matters.

6. **Why does LinearSVC work well on text?**  
   It handles high-dimensional sparse features efficiently.

7. **What is stratification?**  
   Splitting while preserving class proportions in train/test.

8. **What is data leakage?**  
   When information from the test set influences training (e.g., fitting TF-IDF on all data).

9. **How does a Pipeline help?**  
   It chains preprocessing + model and applies them consistently — preventing leakage.

10. **Why isn't accuracy enough?**  
    Imbalanced classes can yield high accuracy with useless positive-class performance.

11. **Precision vs recall?**  
    Precision = correctness of positive predictions. Recall = coverage of actual positives.

12. **What is F1 score?**  
    Harmonic mean of precision and recall.

13. **How do you handle class imbalance?**  
    `class_weight='balanced'`, resampling, better metrics, more representative data.

14. **Word vs character features?**  
    Word features capture meaning; character features handle noise, typos, short text.

15. **Why is error analysis important?**  
    It reveals the *types* of failures (slang, negation) so you know what to fix.

---
## 🏁 23. Key Takeaways

### You now understand

- What text classification is
- Why it's usually supervised
- Binary / multiclass / multi-label
- BoW / TF-IDF → ML models
- Naive Bayes
- Logistic Regression
- Linear SVM
- Train/test split
- Stratification
- Data leakage
- Scikit-learn Pipelines
- Accuracy, Precision, Recall, F1
- Confusion matrix
- Class imbalance
- Word vs character features
- Baseline-first development
- Hyperparameter tuning
- Error analysis
- Real-world applications

### The workflow

```text
Text → TF-IDF → Classifier → Prediction → Evaluation
```

---

### 🔗 What's Next?

**08 — Sentiment Analysis**

We apply text classification specifically to **positive, negative, and neutral opinions/emotions in text**.